# SPOT heatmap simulation ($\alpha=0.5$)

This notebook reproduces the two-dimensional discovery-error heatmap.

In [ ]:
import os
import math
import hashlib
from dataclasses import dataclass
from typing import Dict, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from joblib import Parallel, delayed
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(), "| Device:", DEVICE)

np.random.seed(0)

OUTPUT_DIR = "outputs_discovery_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def seed_from_params(base_seed: int, **kwargs) -> int:
    items = ",".join(f"{key}={kwargs[key]}" for key in sorted(kwargs))
    value = f"{base_seed}|{items}"
    digest = hashlib.blake2b(value.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, "little") & 0xFFFFFFFF


## Simulation model and SPOT parameters

In [ ]:
def sample_theta_iid(
    n: int,
    eps: float,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    return (rng.random(n) < eps).astype(np.int8)


def sample_theta_markov_stationary(
    n: int,
    eps: float,
    s: float = 0.6,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[np.ndarray, float, float]:
    if rng is None:
        rng = np.random.default_rng()
    s = float(np.clip(s, 1e-6, 1.0 - 1e-6))
    eps = float(np.clip(eps, 0.0, 1.0))
    a = eps * s
    b = (1.0 - eps) * s

    theta = np.zeros(n, dtype=np.int8)
    theta[0] = int(rng.random() < eps)
    for t in range(1, n):
        if theta[t - 1] == 0:
            theta[t] = int(rng.random() < a)
        else:
            theta[t] = int(rng.random() >= b)
    return theta, a, b


@dataclass
class SimParams:
    n: int
    alpha_vocab: float
    r_core: float
    p_sparse: float
    q_sing: float
    theta_mode: str = "markov"
    markov_s: float = 0.6
    kappa_core_mass: float = 0.2
    eps_C0: float = 0.5
    pt_mode: str = "hclplus_indepmix"
    oracle_eps: bool = False
    joint_rho: float = 0.95
    joint_amp: float = 0.2
    hcl_cC: float = 0.5
    hcl_CC: float = 2.0


def _markov_chain_Z(
    n: int,
    rho: float = 0.95,
    rng: Optional[np.random.Generator] = None,
) -> np.ndarray:
    if rng is None:
        rng = np.random.default_rng()
    rho = float(np.clip(rho, 1e-6, 1.0 - 1e-6))
    Z = np.empty(int(n), dtype=np.int8)
    Z[0] = int(rng.random() < 0.5)
    u = rng.random(int(n) - 1)
    for t in range(1, int(n)):
        Z[t] = Z[t - 1] if u[t - 1] < rho else 1 - Z[t - 1]
    return Z


def _hclplus_constants(
    n: int,
    alpha: float,
    r_core: float,
    q: float,
    cC: float,
    CC: float,
    c_light_mult: float,
    c_core_mult: float,
) -> Dict[str, float]:
    mL = int(max(1, round(n ** alpha)))
    mC = int(max(1, round(n ** r_core)))

    p_light = float(np.clip(c_light_mult, 0.0, 10.0)) * n ** (-(alpha + q))
    p_core = float(np.clip(c_core_mult, cC, CC)) * n ** (-(r_core + q))

    Delta = float(n ** (-q))
    Delta_core = float(np.clip(mC * p_core, 0.0, 0.9 * Delta))
    Delta_light = float(np.clip(mL * p_light, 0.0, 0.9 * Delta))
    total = Delta_core + Delta_light
    if total <= 0:
        Delta_core = 0.0
        Delta_light = 0.0
    elif total > Delta:
        scale = Delta / total
        Delta_core *= scale
        Delta_light *= scale

    p_heavy = float(np.clip(1.0 - Delta_core - Delta_light, 0.0, 1.0))
    p_core_eff = Delta_core / mC if mC > 0 else 0.0
    p_light_eff = Delta_light / mL if mL > 0 else 0.0

    return {
        "mL": float(mL),
        "mC": float(mC),
        "Delta": Delta,
        "Delta_core": Delta_core,
        "Delta_light": Delta_light,
        "p_heavy": p_heavy,
        "p_core": float(p_core_eff),
        "p_light": float(p_light_eff),
    }


def _make_hclplus_consts_for_Z(params: SimParams, Z_val: int) -> Dict[str, float]:
    amp = float(params.joint_amp)
    z_sign = 2 * int(Z_val) - 1
    c_light_mult = float(np.clip(1.0 + 0.25 * amp * z_sign, 0.7, 1.3))
    c_core_mult = float(np.clip(0.6 + 0.20 * amp * z_sign, 0.3, 1.0))

    return _hclplus_constants(
        n=int(params.n),
        alpha=float(params.alpha_vocab),
        r_core=float(params.r_core),
        q=float(params.q_sing),
        cC=float(params.hcl_cC),
        CC=float(params.hcl_CC),
        c_light_mult=c_light_mult,
        c_core_mult=c_core_mult,
    )


def _sample_Pt_w_from_consts(
    consts: Dict[str, float],
    m: int,
    rng: np.random.Generator,
) -> np.ndarray:
    if m <= 0:
        return np.empty(0, dtype=float)

    u = rng.random(int(m))
    pH = float(consts["p_heavy"])
    Dc = float(consts["Delta_core"])
    pC = float(consts["p_core"])
    pL = float(consts["p_light"])

    out = np.empty(int(m), dtype=float)
    maskH = u < pH
    maskC = (~maskH) & (u < pH + Dc)
    out[maskH] = pH
    out[maskC] = pC
    out[~(maskH | maskC)] = pL
    return out


def generate_pivots_H1(
    params: SimParams,
    rng: Optional[np.random.Generator] = None,
):
    if rng is None:
        rng = np.random.default_rng()

    n = int(params.n)
    eps_n = float(params.eps_C0) * float(n ** (-float(params.p_sparse)))

    seed_theta = int(rng.integers(0, 2**32 - 1))
    rng_theta = np.random.default_rng(seed_theta)

    Z = _markov_chain_Z(n, rho=float(params.joint_rho), rng=rng)

    theta_mode = str(params.theta_mode).lower()
    if theta_mode == "iid":
        theta = sample_theta_iid(n, eps_n, rng=rng_theta)
        a = b = None
    elif theta_mode == "markov":
        theta, a, b = sample_theta_markov_stationary(
            n,
            eps_n,
            s=float(params.markov_s),
            rng=rng_theta,
        )
    else:
        raise ValueError("theta_mode must be 'iid' or 'markov'")

    U = rng.random(n)
    Y = U.copy()
    signal = theta == 1

    if np.any(signal):
        consts0 = _make_hclplus_consts_for_Z(params, 0)
        consts1 = _make_hclplus_consts_for_Z(params, 1)

        mask0 = signal & (Z == 0)
        if np.any(mask0):
            Pt0 = _sample_Pt_w_from_consts(consts0, int(mask0.sum()), rng)
            Y[mask0] = U[mask0] ** Pt0

        mask1 = signal & (Z == 1)
        if np.any(mask1):
            Pt1 = _sample_Pt_w_from_consts(consts1, int(mask1.sum()), rng)
            Y[mask1] = U[mask1] ** Pt1

    info = {
        "Z": Z,
        "eps_n": eps_n,
        "rho_Z": float(params.joint_rho),
        "theta_mode": theta_mode,
        "markov_s": float(params.markov_s) if theta_mode == "markov" else None,
        "theta_a": a,
        "theta_b": b,
        "regime_amp": float(params.joint_amp),
    }
    return Y, theta, info


@dataclass
class Alg1Params:
    # Algorithm 1 is named SPOT in the current manuscript.
    umin: float = 0.005
    umax: float = 0.98
    tau_dagger: float = 0.85
    Mn_power: float = 1.5
    Mn_mult: float = 3.5
    eta_mult: float = 0.0
    eps_clip: float = 1e-8


## Configuration

In [ ]:
C_grid = np.logspace(np.log10(0.01), np.log10(10.0), 60)

# Change only this value to reproduce the other heatmaps in the paper.
alpha_vocab = 0.5
r_core = 0.0

pt_mode = "hclplus_indepmix"
joint_rho = 0.95
eps_C0 = 0.5
BASE_SEED = 12345

alg = Alg1Params()


## Run the $20\times20$ heatmap simulation

In [ ]:
n_heat = 10000
B_heat = 500


def R(a: float, b: float, K: int) -> np.ndarray:
    return np.linspace(a, b, K)


p_grid_hm = R(0.01, 1.0, 20)

W_light = max(1, int(round(n_heat ** alpha_vocab)))
W_core = max(1, int(round(n_heat ** r_core)))
W_n = max(int(W_light + W_core), 2)
q_min = math.log(W_n / (W_n - 1.0)) / math.log(n_heat)
q_grid_hm = R(q_min, 1.0, 20)

n_jobs = -1
prefer_backend = "threads"


def generate_batch(
    params: SimParams,
    B: int,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    Ys = np.empty((B, params.n), dtype=np.float64)
    Thetas = np.empty((B, params.n), dtype=np.int8)
    for b in range(B):
        Y, theta, _ = generate_pivots_H1(params, rng=rng)
        Ys[b] = Y
        Thetas[b] = theta.astype(np.int8)
    return Ys, Thetas


def eval_err_over_C_torch_from_batch(
    params: SimParams,
    alg: Alg1Params,
    C_grid: np.ndarray,
    Y_batch: np.ndarray,
    theta_batch: np.ndarray,
) -> Tuple[float, float]:
    B, n = Y_batch.shape
    Y = torch.as_tensor(Y_batch, dtype=torch.float64, device=DEVICE)
    theta = torch.as_tensor(theta_batch, dtype=torch.int64, device=DEVICE)
    Y_sorted, _ = torch.sort(Y, dim=1)

    M = int(max(20, np.ceil(alg.Mn_mult * np.log(n) ** alg.Mn_power)))
    u_grid = torch.linspace(alg.umin, alg.umax, M, device=DEVICE, dtype=torch.float64)
    n_tensor = torch.tensor(float(n), device=DEVICE, dtype=torch.float64)
    taus_1d = 1.0 - torch.pow(n_tensor, -u_grid)

    taus = taus_1d.view(1, -1).expand(B, -1).contiguous()
    idx = torch.searchsorted(Y_sorted, taus, right=True)
    Shat = (n - idx).to(torch.float64) / float(n)
    S0 = torch.pow(n_tensor, -u_grid).view(1, -1)

    tau_d = float(alg.tau_dagger)
    tau_d_t = torch.full((B, 1), tau_d, device=DEVICE, dtype=torch.float64)
    idx_d = torch.searchsorted(Y_sorted, tau_d_t, right=True).squeeze(1)
    Shat_d = (n - idx_d).to(torch.float64) / float(n)
    S0_d = 1.0 - tau_d
    eps_hat = (Shat_d - S0_d) / max(1e-15, 1.0 - S0_d)
    eps_hat = torch.clamp(eps_hat, 0.0, 1.0).to(torch.float64)

    denom = torch.maximum(Shat, torch.full_like(Shat, 1.0 / float(n)))
    That = (1.0 - eps_hat.unsqueeze(1)) * (S0 / denom)

    logn = float(np.log(n))
    lambda_grid = torch.as_tensor(C_grid, dtype=torch.float64, device=DEVICE) / logn
    eta = float(alg.eta_mult) / logn**2

    cond = That.unsqueeze(1) <= (lambda_grid - eta).view(1, -1, 1)
    any_true = cond.any(dim=2)
    first_idx = torch.argmax(cond.to(torch.int64), dim=2)
    u_idx = torch.where(any_true, first_idx, torch.full_like(first_idx, M - 1))

    tau_hat = taus_1d[u_idx]
    selected = Y.unsqueeze(1) > tau_hat.unsqueeze(2)
    theta_bool = theta.unsqueeze(1).bool()

    TP = (selected & theta_bool).sum(dim=2).to(torch.float64)
    FP = (selected & (~theta_bool)).sum(dim=2).to(torch.float64)
    p_miss = (selected.sum(dim=2) == 0).to(torch.float64).mean(dim=0)

    FP_sum = FP.sum(dim=0)
    TP_sum = TP.sum(dim=0)
    FPR = FP_sum / torch.maximum(FP_sum + TP_sum, torch.ones_like(FP_sum))
    error = FPR + p_miss

    min_error, index = torch.min(error, dim=0)
    return float(min_error.item()), float(C_grid[int(index.item())])


minErr_hm = np.zeros((len(q_grid_hm), len(p_grid_hm)), dtype=float)
Cstar_hm = np.zeros_like(minErr_hm)

for iq, q in enumerate(tqdm(q_grid_hm, desc="q rows")):
    params_list = []
    seed_list = []
    for p in p_grid_hm:
        params = SimParams(
            n=int(n_heat),
            alpha_vocab=float(alpha_vocab),
            r_core=float(r_core),
            p_sparse=float(p),
            q_sing=float(q),
            joint_rho=float(joint_rho),
            eps_C0=float(eps_C0),
            pt_mode=pt_mode,
        )
        params_list.append(params)
        seed_list.append(
            seed_from_params(
                BASE_SEED,
                n=int(n_heat),
                alpha=float(alpha_vocab),
                r=float(r_core),
                p_sparse=float(p),
                q_sing=float(q),
            )
        )

    def generate_for_index(ip: int):
        rng = np.random.default_rng(seed_list[ip])
        Y_batch, theta_batch = generate_batch(params_list[ip], B=B_heat, rng=rng)
        return ip, Y_batch, theta_batch

    batches = Parallel(n_jobs=n_jobs, prefer=prefer_backend)(
        delayed(generate_for_index)(ip) for ip in range(len(p_grid_hm))
    )

    for ip, Y_batch, theta_batch in batches:
        min_error, C_star = eval_err_over_C_torch_from_batch(
            params_list[ip],
            alg,
            C_grid=C_grid,
            Y_batch=Y_batch,
            theta_batch=theta_batch,
        )
        minErr_hm[iq, ip] = min_error
        Cstar_hm[iq, ip] = C_star


## Plot and save the heatmap as PDF

In [ ]:
title_fs = 18
label_fs = 16
tick_fs = 14
clip_max = 1.05
minErr_hm_clip = np.minimum(minErr_hm, clip_max)

fig, ax = plt.subplots(figsize=(7.2, 5.6), dpi=150)
im = ax.imshow(
    minErr_hm_clip,
    origin="lower",
    aspect="auto",
    extent=[p_grid_hm[0], p_grid_hm[-1], q_grid_hm[0], q_grid_hm[-1]],
    vmin=0.0,
    vmax=clip_max,
    interpolation="bilinear",
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r"$\mathrm{Err}_\delta$", fontsize=label_fs)
cbar.ax.tick_params(labelsize=tick_fs)

p_min, p_max = p_grid_hm[0], p_grid_hm[-1]
q_min_plot, q_max = q_grid_hm[0], q_grid_hm[-1]
p_line = np.linspace(p_min, p_max, 400)
q_line = 1.0 - p_line
mask = (q_line >= q_min_plot) & (q_line <= q_max)
ax.plot(p_line[mask], q_line[mask], "w--", linewidth=2, alpha=0.95)
if p_min <= alpha_vocab <= p_max:
    ax.plot([alpha_vocab, alpha_vocab], [q_min_plot, q_max], "w--", linewidth=2, alpha=0.95)

ax.set_xlabel("p", fontsize=label_fs)
ax.set_ylabel("q", fontsize=label_fs)
ax.tick_params(labelsize=tick_fs)
ax.set_title(rf"$\alpha={alpha_vocab:.2f}$", fontsize=title_fs)

fig.tight_layout()
alpha_tag = str(alpha_vocab).replace(".", "p")
pdf_path = os.path.join(OUTPUT_DIR, f"heatmap_alpha{alpha_tag}.pdf")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
print("Saved:", pdf_path)
plt.show()
